# PRACTICE 3: CLUSTERING RETAIL CUSTOMERS
## CHẶNG 3 & 4: HUẤN LUYỆN, ĐÁNH GIÁ MÔ HÌNH GOM CỤM & LẬP HỒ SƠ PHÂN KHÚC

> **Mục tiêu kỹ thuật:**
> - Xác định số cụm $K$ tối ưu toán học qua phương pháp Khuỷu tay (Elbow WCSS) và hệ số Silhouette Score.
> - Huấn luyện bộ 3 thuật toán Scratch: **K-Means**, **Hierarchical Agglomerative** và **DBSCAN**.
> - Đối chuẩn độ chuẩn xác toán học tuyệt đối với Scikit-Learn (Inertia, ARI = 1.0, Điểm nhiễu ngoại lai).
> - Trực quan hóa bản đồ phân cụm 2D, biểu đồ cây Dendrogram (đề bài slide yêu cầu) và phân cụm mật độ DBSCAN.
> - Lập hồ sơ 5 phân khúc khách hàng mục tiêu (Personas) & đề xuất chiến lược bán lẻ cá nhân hóa.

---
## Mục 1: Tối Ưu Hóa Số Cụm K (Elbow Method & Silhouette Analysis)

### 1.1 Khai Báo Thư Viện & Nạp Ma Trận Đặc Trưng
* **Mục tiêu:** Nạp ma trận 2D chuẩn hóa và các thuật toán Scratch từ thư viện `models.algorithms`.

In [2]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from scripts.feature_engineering import prepare_clustering_features
from models.algorithms.k_means import KMeansClustering
from models.algorithms.hierarchical import HierarchicalClustering
from models.algorithms.dbscan import DBSCANClustering
from scripts.model_training import find_optimal_k_elbow, find_optimal_k_silhouette, tune_dbscan_params

X_scaled, scaler_obj, feature_names = prepare_clustering_features(feature_set='2d')
print(f"✔ Đã nạp không gian phân cụm 2D: {feature_names} — Shape: {X_scaled.shape}")

✔ Đã nạp không gian phân cụm 2D: ['Annual_Income_k$', 'Spending_Score'] — Shape: (200, 2)


### 1.2 Quét Tìm K Tối Ưu Qua Phương Pháp Khuỷu Tay (Elbow) & Silhouette Score
* **Mục tiêu:** Tính toán độ uốn quán tính WCSS và hệ số Silhouette với K từ 2 đến 10.

In [3]:
k_range = range(2, 11)
elbow_scores = find_optimal_k_elbow(X_scaled, k_range=k_range)
sil_scores = find_optimal_k_silhouette(X_scaled, k_range=k_range)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Đồ thị Elbow
axes[0].plot(list(elbow_scores.keys()), list(elbow_scores.values()), 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=5, color='red', linestyle='--', label='Điểm uốn tối ưu (K=5)')
axes[0].set_title("Phương pháp Khuỷu tay (Elbow Method: K vs WCSS)", fontsize=11, fontweight='bold')
axes[0].set_xlabel("Số cụm K")
axes[0].set_ylabel("Inertia (WCSS)")
axes[0].legend()

# Đồ thị Silhouette Score
axes[1].plot(list(sil_scores.keys()), list(sil_scores.values()), 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=5, color='red', linestyle='--', label='Đỉnh Silhouette cực đại (K=5)')
axes[1].set_title("Đồ thị Hệ số Silhouette Score theo K", fontsize=11, fontweight='bold')
axes[1].set_xlabel("Số cụm K")
axes[1].set_ylabel("Silhouette Score")
axes[1].legend()

plt.tight_layout()
plt.show()

best_k = max(sil_scores, key=lambda k: sil_scores[k])
print(f"✔ Số cụm K tối ưu toán học: K = {best_k} (Silhouette đạt cực đại {sil_scores[best_k]})")

✔ Số cụm K tối ưu toán học: K = 5 (Silhouette đạt cực đại 0.5547)


---
## Mục 2: Huấn Luyện Bộ 3 Thuật Toán Scratch & Đối Chuẩn Sklearn

### 2.1 Huấn Luyện K-Means Scratch (K=5) & So Sánh Sklearn
* **Mục tiêu:** Chạy thuật toán K-Means Scratch khởi tạo k-means++, đối chuẩn Inertia và nhãn với Scikit-Learn.

In [4]:
from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.metrics import adjusted_rand_score

km_scratch = KMeansClustering(n_clusters=5, random_state=42)
labels_km = km_scratch.fit_predict(X_scaled)

km_sklearn = SklearnKMeans(n_clusters=5, random_state=42, n_init=10)
labels_sklearn = km_sklearn.fit_predict(X_scaled)

print(f"• Inertia K-Means Scratch : {km_scratch.inertia_:.8f}")
print(f"• Inertia Sklearn KMeans  : {km_sklearn.inertia_:.8f}")
print(f"• Độ sai lệch Inertia     : {abs(km_scratch.inertia_ - km_sklearn.inertia_):.2e}")
print(f"• Chỉ số tương đồng ARI   : {adjusted_rand_score(labels_km, labels_sklearn):.4f}")

• Inertia K-Means Scratch : 65.56840816
• Inertia Sklearn KMeans  : 65.56840816
• Độ sai lệch Inertia     : 1.42e-14
• Chỉ số tương đồng ARI   : 1.0000


### 2.2 Huấn Luyện Hierarchical Clustering Scratch (Ward Linkage) & Biểu Đồ Cây Dendrogram
* **Mục tiêu:** Cắt cây phân cấp tại $K=5$ và vẽ biểu đồ Dendrogram theo đúng yêu cầu đề bài slide.

In [5]:
from scipy.cluster.hierarchy import dendrogram

hc_scratch = HierarchicalClustering(n_clusters=5, linkage='ward')
labels_hc = hc_scratch.fit_predict(X_scaled)

plt.figure(figsize=(12, 6))
dendrogram(hc_scratch.linkage_matrix_, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=10, show_contracted=True)
plt.axhline(y=1.5, color='red', linestyle='--', label='Đường cắt K = 5 cụm (d ≈ 1.5)')
plt.title("Biểu đồ Cây Phân cấp Dendrogram (Hierarchical Clustering - Ward Linkage)", fontsize=12, fontweight='bold')
plt.xlabel("Chỉ số cụm gộp")
plt.ylabel("Khoảng cách khoảng cách Ward")
plt.legend()
plt.show()

### 2.3 Huấn Luyện DBSCAN Scratch & Cô Lập Điểm Nhiễu Ngoại Lai
* **Mục tiêu:** Tìm kiếm mật độ với eps=0.35, min_samples=5 để nhận diện các cụm mật độ và cô lập các mẫu ngoại lai.

In [6]:
db_scratch = DBSCANClustering(eps=0.35, min_samples=5)
labels_db = db_scratch.fit_predict(X_scaled)

print(f"• Số cụm mật độ phát hiện: {db_scratch.n_clusters_} cụm.")
print(f"• Số điểm nhiễu ngoại lai (nhãn -1): {db_scratch.n_noise_} khách hàng ({round(db_scratch.n_noise_/len(X_scaled)*100, 1)}%).")

• Số cụm mật độ phát hiện: 6 cụm.
• Số điểm nhiễu ngoại lai (nhãn -1): 23 khách hàng (11.5%).


---
## Mục 3: Bản Đồ Phân Cụm 2D & Bảng Hồ Sơ 5 Phân Khúc Khách Hàng

### 3.1 Bản Đồ Phân Cụm 2D Không Gian Khách Hàng (K-Means)
* **Mục tiêu:** Trực quan hóa 5 phân khúc khách hàng cùng các tâm cụm (centroid).

In [7]:
from scripts.common import PROCESSED_CSV
df_clean = pd.read_csv(PROCESSED_CSV)

plt.figure(figsize=(10, 6))
colors = ['#e53e3e', '#3182ce', '#38a169', '#d69e2e', '#805ad5']
cluster_names = [
    'Cụm 0: Tiêu xài Tự do (Spenders)',
    'Cụm 1: Khách hàng VIP (Target)',
    'Cụm 2: Khách hàng Tiết kiệm (Sensible)',
    'Cụm 3: Cẩn trọng / Giữ tiền (Careful)',
    'Cụm 4: Khách hàng Tiêu chuẩn (Standard)'
]

for c in range(5):
    mask = labels_km == c
    plt.scatter(df_clean.loc[mask, 'Annual_Income_k$'], df_clean.loc[mask, 'Spending_Score'],
                color=colors[c], s=70, alpha=0.85, edgecolors='white', linewidth=0.8, label=cluster_names[c])

# Đưa tâm cụm về thang đo gốc
centers_orig = scaler_obj.inverse_transform(km_scratch.cluster_centers_)
plt.scatter(centers_orig[:, 0], centers_orig[:, 1], c='black', marker='X', s=200, edgecolors='white', linewidth=2, label='Tâm cụm Centroids')

plt.title("Bản đồ Phân cụm Khách hàng 2D K-Means (Annual Income vs Spending Score)", fontsize=12, fontweight='bold')
plt.xlabel("Thu nhập hàng năm (Annual Income k$)", fontsize=11)
plt.ylabel("Điểm chi tiêu (Spending Score 1-100)", fontsize=11)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

### 3.2 Bảng Hồ Sơ 5 Phân Khúc Khách Hàng & Chiến Lược Bán Lẻ
* **Mục tiêu:** Xây dựng bảng đặc tính kinh doanh và khuyến nghị hành động chiến lược bán lẻ cho từng nhóm persona.

In [8]:
from scripts.model_evaluation import profile_customer_segments

df_profile = profile_customer_segments(df_clean, labels_km)
display(df_profile[['Cluster', 'Tên phân khúc (Persona)', 'Số lượng', 'Tỷ lệ (%)', 'Thu nhập TB (k$)', 'Chi tiêu TB', 'Tuổi TB', 'Chiến lược Bán lẻ']])

,Cluster,Tên phân khúc (Persona),Số lượng,Tỷ lệ (%),Thu nhập TB (k$),Chi tiêu TB,Tuổi TB,Chiến lược Bán lẻ
0,0,Nhóm Tiêu xài Tự do (Spenders),81,40.5%,55.3,49.5,42.7,Thu nhập thấp nhưng chi tiêu mạnh. Đẩy mạnh cá...
1,1,Nhóm Khách hàng VIP (Target Customers),35,17.5%,88.2,17.1,41.1,Thu nhập cao và chi tiêu cực mạnh. Chăm sóc đặ...
2,2,Nhóm Khách hàng Tiết kiệm (Sensible),22,11.0%,25.7,79.4,25.3,Thu nhập thấp và chi tiêu dè dặt. Cung cấp các...
3,3,Nhóm Cẩn trọng / Giữ tiền (Careful / Savers),39,19.5%,86.5,82.1,32.7,Thu nhập cao nhưng chi tiêu rất thấp. Cần tiếp...
4,4,Nhóm Khách hàng Tiêu chuẩn (Standard / Average),23,11.5%,26.3,20.9,45.2,Thu nhập và chi tiêu đều ở mức trung bình (chi...
